In [130]:
%load_ext autoreload
%autoreload 2

import warnings
from pandas.errors import SettingWithCopyWarning

warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from app.logger import *
import json5,json
import fitz #type: ignore

from app.amc.fund_data import *
from app.utils import *
from app.konstant import get_config, get_regex

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [134]:
amc_id = '59_0'
path = r"C:\Users\kaustubh.keny\Downloads\AMC\59_30-Nov-25_FS.pdf"
config = get_config("2025",amc_id)
regex = get_regex("2025")

object = BajajFinServ(config,regex,path)
title,path_pdf= object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\mywork-repo\config\2025\59_0_AMC.json5
Col 1: [0], Col 2: [5]
['bajaj finserv flexi cap fund', 'Beta', '0.84', 'Banks', '16.25%']
['', 'Sharpe ratio', '1.20', 'Pharmaceuticals & Biotechnology', '10.43%']
['', 'Jensen’s alpha', '4.81%', 'Finance', '9.14%']
['', 'Standard Deviation', '12.25%', '', '']
['', 'Information ratio', '0.64', '', '']
['bajaj finserv large and mid cap fund', 'Beta', '0.77', 'Banks', '18.06%']
['', 'Sharpe ratio', '0.63', 'Pharmaceuticals & Biotechnology', '11.65%']
['', 'Jensen’s alpha', '2.90%', 'Finance', '6.91%']
['', 'Standard Deviation', '12.40%', '', '']
['', 'Information ratio', '0.29', '', '']
['bajaj finserv large cap fund', 'Beta', '0.91', 'Banks', '23.05%']
['', 'Sharpe ratio', '-0.30', 'Pharmaceuticals & Biotechnology', '8.27%']
['', 'Jensen’s Alpha', '-1.21%', 'IT - Software', '7.15%']
['', 'Standard deviation', '12.13%', '', '']
['', 'Information ratio', '-0.31', '', '']
['bajaj finserv consumption fun

In [136]:
config = get_config("2025",amc_id)
regex = get_regex("2025")
object = Union(config,regex,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

[special] No method found for _update_bench_data
[special] No method found for _update_date_data
[special] No method found for _update_bench_data
[special] No method found for _update_date_data
[special] No method found for _update_bench_data
[special] No method found for _update_date_data
[special] No method found for _update_bench_data
[special] No method found for _update_date_data
[special] No method found for _update_bench_data
[special] No method found for _update_date_data
[special] No method found for _update_bench_data
[special] No method found for _update_date_data
[special] No method found for _update_bench_data
[special] No method found for _update_date_data
[special] No method found for _update_bench_data
[special] No method found for _update_date_data
[special] No method found for _update_bench_data
[special] No method found for _update_date_data
[special] No method found for _update_bench_data
[special] No method found for _update_date_data
[special] No method found for 

C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\mywork-repo\config\2025\59_0_AMC.json5


In [121]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\40_30-Nov-25_FS.json


In [ ]:
pattern = "([\\w\\s]+?)\\s*(?:Over|Around)\\s*([0-9]+\\s*years).*?\\s*Managing this\\s+scheme\\s+since\\s+([A-Za-z]+\\s*\\d{1,2},?\\s*\\d{4}|[Ii]nception)"
for fund, content in final_text.items():
    # check = 'before.fund_manager'
    for key in content:
        if key.endswith("fund_managers"):
            print(fund)
            text =re.sub("[^A-Za-z0-9\\s\\-\\(\\)\\.\\,\\+\\%\\:\\&]+", "",content[key]).strip()
            print(text)
            match = re.findall(pattern,text, re.IGNORECASE)
            print(match)

In [ ]:
import os, json
import pandas as pd


"Helios Capital Asset Management  (India) Private Limited"

static_keys = [
        "amc_name", "main_scheme_name", "mutual_fund_name", "benchmark_index", "monthly_aaum_date", 
        "monthly_aaum_value", "scheme_launch_date", "min_addl_amt", "min_addl_amt_multiple", 
        "min_amt", "min_amt_multiple",
    ]
load_keys = ["entry","exit"]
metric_keys = [
    "alpha", "arithmetic_mean_ratio", "average_div_yield", "average_pb", "average_pe", "avg_maturity",
    "beta", "correlation_ratio", "downside_deviation", "information_ratio", "macaulay",
    "mod_duration", "port_turnover_ratio", "r_squared_ratio", "roe_ratio", "sharpe", "sortino_ratio",
    "std_dev", "tracking_error", "treynor_ratio", "upside_deviation", "ytm"
]

manager_keys = ["name","managing_fund_since","total_exp","qualification"]

def flatten_to_row(value, max_manager):
    add_value = []

    # Static keys
    for k in static_keys:
        val = value.get(k, "")
        if isinstance(val, list):
            val = ", ".join(val)
        add_value.append(val)

    # Loads
    entry, exit = "", ""
    for l in value.get("load", []):
        if l.get("type") == "entry":
            entry = l.get("comment", "")
        elif l.get("type") == "exit":
            exit = l.get("comment", "")
    add_value.extend([entry, exit])

    # Metrics
    metric_map = {m["name"]: m["value"] for m in value.get("metrics", [])}
    metric = [metric_map.get(k, "") for k in metric_keys]
    add_value.extend(metric)

    # Fund managers (pad to max_manager)
    managers = value.get("fund_manager", [])
    for i in range(max_manager):
        if i < len(managers):
            fm = managers[i]
            add_value.extend([
                fm.get("name", ""),
                fm.get("managing_fund_since", ""),
                fm.get("total_exp", ""),
                fm.get("qualification", "")
            ])
        else:
            # pad with blanks if fewer managers
            add_value.extend(["", "", "", ""])

    return add_value


path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\12_31-Oct-25_FS.json"
with open(path,"r",encoding="utf8") as f:
    df = json.load(f)
sheet_name = df.get("metadata",{}).get("document_name","")
records = df.get("records",[])
max_fund_managers = max(
    (len(record["value"].get("fund_manager", [])) for record in records),
    default=0
)


headers = static_keys + load_keys + metric_keys
for i in range(1, max_fund_managers + 1):
    headers.extend([f"{key}_{i}" for key in manager_keys])

file_name = path.split("\\")[-1].replace(".json","")

rows = [flatten_to_row(record["value"], max_fund_managers) for record in records]
df_out = pd.DataFrame(rows, columns=headers)
df_out.to_csv(f"{file_name}.csv", index=False)



3
